In [1]:
# %% [Step 1: Import libraries and load dataset]

import pandas as pd

# Load dataset (updated path as requested)
file_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_NPV_IRR_update.csv"
df = pd.read_csv(file_path)

# Preview dataset
df.head()

,BUILDING_REFERENCE_NUMBER,OSG_REFERENCE_NUMBER,ADDRESS1,ADDRESS2,ADDRESS3,POSTCODE,LATITUDE,LONGITUDE,ORIENTATION,ORIENTATION_ESPR_ROT,...,PV_NPV_FLEX,BATTERY_NPV_FLEX,WINDOWS_IRR_FLEX,ROOF_IRR_FLEX,WALLS_IRR_FLEX,FAB_IRR_FLEX,HP_IRR_FLEX,SOLAR_THERMAL_IRR_FLEX,PV_IRR_FLEX,BATTERY_IRR_FLEX
0,1001829067,122009933.0,19 CULCREUCH AVENUE,FINTRY,GLASGOW,G63 0YB,56.0557,-4.2233,100,170,...,0.000000,-4836.000000,NaN,NaN,NaN,NaN,0.227522,0.105684,NaN,NaN
1,1001951858,122010025.0,GLENVIEW,FINTRY,GLASGOW,G63 0XJ,56.0528,-4.2206,180,90,...,1962.148428,-2558.147057,-0.015884,-0.074175,-0.017873,-0.009717,0.418858,0.137610,0.081951,-0.037370
2,1001829071,122009868.0,22 CULCREUCH AVENUE,FINTRY,GLASGOW,G63 0YB,56.0555,-4.2237,100,170,...,-2000.000000,-4836.000000,0.115626,1.741603,NaN,NaN,NaN,-0.040909,NaN,NaN
3,1234761001,122009968.0,1 MENZIES TERRACE,FINTRY,GLASGOW,G63 0YJ,56.0584,-4.2248,150,120,...,1179.292669,-2385.903238,-0.079761,-0.058603,-0.003530,0.003958,-0.012701,0.006844,0.073403,-0.029204
4,1001991633,122009892.0,50 MAIN STREET,FINTRY,GLASGOW,G63 0XF,56.0547,-4.2283,150,120,...,-21000.000000,-4836.000000,0.372136,2.422694,NaN,NaN,NaN,0.129009,NaN,NaN


In [2]:
# %% [Step 2: Inspect key columns]

# Check built form distribution
df["BUILT_FORM"].value_counts()

# Check extension count distribution (important for new logic)
df["EXTENSION_COUNT"].value_counts()

EXTENSION_COUNT
0    80
1    28
2     6
4     2
3     1
Name: count, dtype: int64

In [3]:
# %% [Step 3: Define archetype assignment function with extension logic]

def assign_archetype(row):
    built_form = str(row["BUILT_FORM"]).strip().lower()
    floors = row["NUMBER_OF_FLOORS"]
    
    # Safely handle extension count (default to 0 if missing/NaN)
    ext_count = row.get("EXTENSION_COUNT", 0)
    if pd.isna(ext_count):
        ext_count = 0

    # --- Original archetype logic ---
    # Semi-detached houses (check first to avoid catching "detached")
    if "semi-detached" == built_form:
        base = f"SemiDetached_{floors}F"

    # Detached houses
    elif "detached" == built_form:
        base = f"Detached_{floors}F"

    # End terrace
    elif "end-terrace" == built_form or "enclosed end-terrace" == built_form:
        base = f"EndTerrace_{floors}F"

    # Mid terrace
    elif "mid-terrace" == built_form:
        base = f"MidTerrace_{floors}F"

    # Flats / apartments
    elif "flat" in built_form:
        base = f"Flat_{floors}F"

    # Fallback category
    else:
        base = f"Other_{floors}F"

    # --- New extension logic ---
    try:
        ext_count = int(ext_count)
    except:
        ext_count = 0

    if ext_count > 0:
        return f"{base}_{ext_count}"
    else:
        return base

In [4]:
# %% [Step 4: Apply archetype assignment]

df["ARCHETYPE"] = df.apply(assign_archetype, axis=1)

In [5]:
# %% [Step 5: Validate results]

# Distribution of new archetypes
df["ARCHETYPE"].value_counts()

# Spot check
df[["BUILT_FORM", "NUMBER_OF_FLOORS", "EXTENSION_COUNT", "ARCHETYPE"]].head(20)

,BUILT_FORM,NUMBER_OF_FLOORS,EXTENSION_COUNT,ARCHETYPE
0,Semi-Detached,2,0,SemiDetached_2F
1,Detached,2,0,Detached_2F
2,End-Terrace,2,0,EndTerrace_2F
3,Detached,2,0,Detached_2F
4,Detached,1,0,Detached_1F
5,Semi-Detached,2,0,SemiDetached_2F
6,Semi-Detached,2,0,SemiDetached_2F
7,Detached,2,2,Detached_2F_2
8,Detached,2,1,Detached_2F_1
9,Detached,3,0,Detached_3F


In [6]:
# %% [Step 6: Save updated dataset]

output_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_NPV_IRR_update2.csv"
df.to_csv(output_path, index=False)